<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">05 Mobile AI</h1>
<hr>

## **17. Local Agents II: Vibe Coding**

<div class="alert alert-light">

  **Name:** (add your name here)

  **Matriculation Number:** (add your number here)

</div>

### 17.1 Introduction

In [05.15_Local_Agents_I.ipynb](05.15_Local_Agents_I.ipynb) you met agents: a model, some tools and a loop. This notebook points that idea at writing code. In February 2025, Andrej Karpathy described a new way of programming: you say what you want, a model writes the code, you run it, and you steer with your next sentence instead of reading every line [01]. He called it **vibe coding**, and by the end of the year Collins Dictionary had chosen it as its word of the year.

Vibe coding is not the same as asking a model a question. When you prompt a model, you read the answer and judge it yourself. When you vibe code, the answer is a program: you judge it by running it, and what happens when it runs becomes your next prompt. Coding agents go one step further and edit files and run commands on their own.

In this notebook you build the vibe coding loop yourself, one piece at a time, on the Pi. Then a second model reviews the code before it runs, you measure why coding agents are hard on a small device, and you connect the Pi to VS Code. The notebook ends with where vibe coding stands today: what it is good at, what goes wrong, and what changes when the model runs locally. The rules for working with any coding assistant are in [00.4_Vibe_Coding_Best_Practices.ipynb](../00_Intro/00.4_Vibe_Coding_Best_Practices.ipynb); this notebook does not repeat them.

Allow about **75 minutes**, plus time for the questions.

By the end, you should be able to explain how vibe coding differs from prompting, build a loop in which a model writes, tests and fixes code, use a second model as a reviewer, estimate whether a coding agent fits on a device, and judge where local vibe coding is worth it.

#### 17.1.1 Environment Setup

Sections 17.2 to 17.4 run on your Pi through `caspi`, connected as in [05.04_GettingStarted_with_Raspberry_Pi.ipynb](05.04_GettingStarted_with_Raspberry_Pi.ipynb). Section 17.5 needs VS Code and one extension. The code the model writes runs on your laptop, so read each function before you run it. They are all small.

<div class="alert alert-block alert-success">

If a library is missing, use the installation instructions in [00.1_Check_Course_Environment.ipynb](../00_Intro/00.1_Check_Course_Environment.ipynb). Select that Python environment as this notebook's kernel.
</div>

In [ ]:
import math
import re

import caspi

caspi.connect()

### 17.2 Vibe Coding by Hand

A vibe coding tool does four things over and over: it asks a model for code, takes the code out of the answer, runs it, and tells the model what went wrong. We build each step on its own, then put them together.

We use `qwen2.5-coder:7b` as the writer, the coding model on the Pi. In our runs, general models such as `granite4.2:3b` failed most of the tasks below. The writer is also one of the largest models on the Pi, so the next cell first frees the memory with `caspi.stop()`, as in 05.04, and then loads the writer. Loading takes about three and a half minutes; after that, an answer takes between ten seconds and a minute. The system prompt asks for short code only, which keeps the answers short.

In [ ]:
WRITER = "qwen2.5-coder:7b"
SYSTEM = "You write Python. Reply with one short Python code block: no docstring, no comments, no explanation."

caspi.stop()   # frees the memory: the writer needs most of it
caspi.load(WRITER)

#### 17.2.1 Ask for code

The first task is easy: convert a signal strength in dBm into milliwatts. Watch the answer arrive.

In [ ]:
TASK = "Write a function dbm_to_mw(dbm) that converts a signal strength in dBm to milliwatts."

reply = caspi.chat(WRITER, [{"role": "user", "content": TASK}], system=SYSTEM, think=False, temperature=0)

#### 17.2.2 Take the code out

The answer is text. Code usually sits between two lines of three backticks, with `python` after the first. A regular expression finds it, and if there are no backticks we take the whole answer:

In [ ]:
def code_in(text):
    """The code between ``` lines, or the whole text if there are none."""
    match = re.search(r"```[a-z]*\n(.*?)```", text, re.DOTALL)
    return match.group(1) if match else text


code = code_in(reply.text)
print(code)

#### 17.2.3 Run it

`exec` runs a string as Python code. The function the model wrote then exists in the dictionary `space`, and we can call it. This is the step that makes vibe coding different from prompting, and also the dangerous one: `exec` runs whatever the model wrote, with your rights on your laptop. Here it is one small function you have just read.

In [ ]:
space = {}
exec(code, space)
space["dbm_to_mw"](20)

20 dBm is 100 mW. One example is not a test, though.

#### 17.2.4 Test it

A test is a list of inputs with the outputs we expect. `test` runs the code, tries every case and returns one line for each case that went wrong, or `None` if everything passed. Floating-point results only have to be close, not exactly equal. Tests like these are also how the research on code models measures whether generated code works [06].

In [ ]:
def same(result, expected):
    if isinstance(expected, float):
        return isinstance(result, (int, float)) and math.isclose(result, expected, rel_tol=1e-6)
    return result == expected


def test(code, name, cases):
    """What went wrong, one line per failed case, or None if the function passes every case."""
    space = {}
    try:
        exec(code, space)
    except Exception as error:
        return f"{type(error).__name__}: {error}"
    problems = []
    for args, expected in cases:
        call = f"{name}({', '.join(map(repr, args))})"
        try:
            result = space[name](*args)
        except Exception as error:
            problems.append(f"{call} raised {type(error).__name__}: {error}")
            continue
        if not same(result, expected):
            problems.append(f"{call} returned {result!r}, but {expected!r} was expected")
    return "\n".join(problems) or None


DBM_CASES = [((0,), 1.0), ((20,), 100.0), ((-30,), 0.001)]
print(test(code, "dbm_to_mw", DBM_CASES) or "all tests pass")

#### 17.2.5 Tell the model what went wrong

The second task has a trap. Wi-Fi channels in the 2.4 GHz band are 5 MHz apart, from channel 1 at 2412 MHz to channel 13 at 2472 MHz. Channel 14, used only in Japan, sits at 2484 MHz, 12 MHz above channel 13. The task does not mention it, but the tests do:

In [ ]:
CHANNEL_TASK = "Write a function channel_of(mhz) that returns the Wi-Fi channel number of a frequency in the 2.4 GHz band."
CHANNEL_CASES = [((2412,), 1), ((2437,), 6), ((2472,), 13), ((2484,), 14)]

history = [{"role": "user", "content": CHANNEL_TASK}]
reply = caspi.chat(WRITER, history, system=SYSTEM, think=False, temperature=0)
problem = test(code_in(reply.text), "channel_of", CHANNEL_CASES)
print(problem or "all tests pass")

If tests failed, their messages are exactly what the model needs to hear. We add the model's answer and the problem to the conversation and ask again, as in the chat of [05.04](05.04_GettingStarted_with_Raspberry_Pi.ipynb):

In [ ]:
if problem:
    history += [reply.message, {"role": "user", "content": f"The tests failed:\n{problem}\nFix the function."}]
    reply = caspi.chat(WRITER, history, system=SYSTEM, think=False, temperature=0)
    print(test(code_in(reply.text), "channel_of", CHANNEL_CASES) or "all tests pass")
else:
    print("The first version already passed, so there is nothing to fix.")

The model did not see the tests, only the messages about the ones that failed. Feeding the result of running the code back to the model is what lets it correct itself [02], and it is the core of every coding agent.

Did the second version get channel 14 right? In our runs it did not: the model fixed the formula for channels 1 to 13, but still answered 15 for 2484 MHz. The error message says what is wrong, not why, and the model cannot know that channel 14 breaks the pattern unless someone tells it.

<div class="alert alert-block alert-info">

Write one sentence that gives the model the fact it is missing, and send it with the next cell. How few words are enough?
</div>

In [ ]:
HINT = "Try again."   # change this

history += [reply.message, {"role": "user", "content": HINT}]
reply = caspi.chat(WRITER, history, system=SYSTEM, think=False, temperature=0)
print(test(code_in(reply.text), "channel_of", CHANNEL_CASES) or "all tests pass")

The model writes the code, but the facts come from you. That part of vibe coding does not go away.

#### 17.2.6 Your loop

<div class="alert alert-block alert-info">

Put the pieces together. Fill in `vibe` so that it asks the model, takes the code out, tests it, returns the code when the tests pass, and otherwise sends the problem back, for at most `rounds` rounds. Every piece you need is in the cells above.
</div>

In [ ]:
def vibe(task, name, cases, rounds=3):
    """Let the model write a function until it passes the tests, for at most `rounds` rounds."""
    history = [{"role": "user", "content": task}]
    for attempt in range(1, rounds + 1):
        print(f"--- round {attempt}")
        # 1. ask the model, with the whole history, so it sees its earlier attempts
        # 2. take the code out of the reply and test it
        # 3. if the tests pass, return the code
        # 4. otherwise add the reply and the problem to the history
    print(f"no working function after {rounds} rounds")
    return None

<details>
<summary>Show one way to write it</summary>

```python
def vibe(task, name, cases, rounds=3):
    """Let the model write a function until it passes the tests, for at most `rounds` rounds."""
    history = [{"role": "user", "content": task}]
    for attempt in range(1, rounds + 1):
        print(f"--- round {attempt}")
        reply = caspi.chat(WRITER, history, system=SYSTEM, think=False, temperature=0)
        code = code_in(reply.text)
        problem = test(code, name, cases)
        if problem is None:
            print("all tests pass")
            return code
        print("tests failed:", problem)
        history += [reply.message, {"role": "user", "content": f"The tests failed:\n{problem}\nFix the function."}]
    print(f"no working function after {rounds} rounds")
    return None
```
</details>

Try your loop on a task from [05.12](05.12_Understanding_Inference_Control.ipynb): reading a line of the café router's log. The device name can have two words, which is where simple solutions break.

In [ ]:
LOG_TASK = ("Write a function parse_log(line) for lines like '12:00:07 laptop joined on channel 6, signal -63 dBm'. "
            "It returns a dict with the keys time, device, channel and signal; channel and signal are integers.")
LOG_CASES = [
    (("12:00:07 laptop joined on channel 6, signal -63 dBm",),
     {"time": "12:00:07", "device": "laptop", "channel": 6, "signal": -63}),
    (("12:01:30 card reader joined on channel 11, signal -71 dBm",),
     {"time": "12:01:30", "device": "card reader", "channel": 11, "signal": -71}),
]

log_code = vibe(LOG_TASK, "parse_log", LOG_CASES)

In our runs, the writer split the line at the spaces, counted the words wrong, and sent back the same code in every round. At temperature 0 the same conversation gives the same answer, so a loop can get stuck: the error messages say what is wrong, but the model has no new idea.

<div class="alert alert-block alert-info">

If your loop got stuck too, read the code it wrote. Add one sentence to `LOG_TASK` that tells the model what it keeps getting wrong, and run the cell again.
</div>

#### 17.2.7 Say it in few words

<div class="alert alert-block alert-info">

Your loop is only as good as the task you give it. Write a task that makes the model produce `payoff(me, other)`, the points of one round of the game from [04.4](../04_Mobile_Algorithms/04.4_Tit-for-Tat_Spectrum_Management.ipynb), in at most 20 words. The model gets one round only, so the task has to say everything. Who in your group needs the fewest words, and why does it work?
</div>

In [ ]:
MY_TASK = "Write a function payoff(me, other)."   # change this
PAYOFF_CASES = [(("C", "C"), 3), (("D", "C"), 5), (("C", "D"), 0), (("D", "D"), 1)]

words = len(MY_TASK.split())
solved = vibe(MY_TASK, "payoff", PAYOFF_CASES, rounds=1) is not None
print(f"{words} words:", "solved" if solved and words <= 20 else "not yet")

### 17.3 A Second Pair of Eyes

Tests only check what they test. A function can pass every case and still break on the next input, or do something dangerous. In software teams, a second person reads the code before it goes in. Here that person is a second model, and it has to be small: the writer fills most of the Pi's memory, and a second large model next to it would freeze the Pi (05.04). We use `qwen3.5:0.8b`, a general model rather than a coding one. It loads in about a minute.

In [ ]:
REVIEWER = "qwen3.5:0.8b"
REVIEW = ("You review Python code before it runs. List at most three real problems: bugs, missing cases "
          "or security risks, one line each. If there are none, say so.")

caspi.load(REVIEWER)

#### 17.3.1 Tests pass, review fails

Here is a function that someone vibe coded to average the signal strengths in a log. It passes its test:

In [ ]:
SUSPICIOUS = '''
def average_signal(lines):
    values = [eval(line.split("signal")[1].replace("dBm", "")) for line in lines]
    return sum(values) / len(values)
'''

LINES = ["12:00:07 laptop joined on channel 6, signal -60 dBm", "12:00:09 phone joined on channel 1, signal -70 dBm"]
print(test(SUSPICIOUS, "average_signal", [((LINES,), -65.0)]) or "all tests pass")

<div class="alert alert-block alert-info">

Before you ask the reviewer: what is wrong with this function? Look at `eval`, and think of an empty log.
</div>

In [ ]:
caspi.ask(REVIEWER, SUSPICIOUS, system=REVIEW, think=False, temperature=0, num_predict=200)

`eval` runs whatever text it gets as Python, so a log line written by an attacker would run on your machine, and an empty log divides by zero. Neither shows up in the test. Did the reviewer find them? Did it also report problems that are not real? A reviewer is a model too: it can miss things, and it can raise false alarms.

#### 17.3.2 The reviewer in the loop

<div class="alert alert-block alert-info">

Write `vibe_reviewed`: first run `vibe`, then let the reviewer read the working code, then give the writer one more round with the review added to the task. Run it on the log task. Does the code get better, or only longer?
</div>

In [ ]:
def vibe_reviewed(task, name, cases, rounds=3):
    """vibe, then a review, then one more round that takes the review into account."""
    code = vibe(task, name, cases, rounds)
    # your review step here
    return code


reviewed_code = vibe_reviewed(LOG_TASK, "parse_log", LOG_CASES)

<details>
<summary>Show one way to write it</summary>

```python
def vibe_reviewed(task, name, cases, rounds=3):
    """vibe, then a review, then one more round that takes the review into account."""
    code = vibe(task, name, cases, rounds)
    if code is None:
        return None
    print("--- review")
    review = caspi.ask(REVIEWER, code, system=REVIEW, think=False, temperature=0, num_predict=200).text
    task = f"{task}\n\nThis version passes the tests:\n{code}\nA reviewer found:\n{review}\nFix the real problems."
    return vibe(task, name, cases, rounds=1) or code
```
</details>

The tests still have the last word: a review that makes the code fail the tests is ignored. Writer and reviewer are two agents with different roles, the simplest form of a multi-agent system. Larger coding agents split the work further, into planners, writers, testers and reviewers.

### 17.4 Why Coding Agents Are Hard on a Pi

Our loop sends a few hundred tokens per round. A coding agent in VS Code sends much more in every request: its instructions, the descriptions of its tools, and the files it is working on, often thousands of tokens. Two numbers decide whether that works on the Pi: how fast the model reads, and how much memory a long context needs.

Our writer, `qwen2.5-coder:7b`, is a 4.7 GB file.

#### 17.4.1 Reading speed

<div class="alert alert-block alert-info">

Predict: the next cell gives the writer the first 3000 characters of `caspi.py`, about 800 tokens of real code, and asks for a single token back. How long does the Pi need to read them?
</div>

In [ ]:
source = open("caspi.py", encoding="utf-8").read()[:3000]
print("reading, this takes about a minute ...")

stats = caspi.ask(WRITER, source, num_predict=1, show=False).stats
speed = stats["prompt_tokens_per_s"]
print(f"read {stats['prompt_tokens']} tokens in {stats['prompt_seconds']:.0f} s, {speed:.0f} tokens per second")
for tokens in (2000, 10000, 30000):
    print(f"a request of {tokens:6d} tokens: {tokens / speed / 60:5.1f} minutes before the first word of the answer")

A coding agent sends its whole request again after every action. Ollama keeps the start of the previous request in memory and only reads what is new (05.04), which helps, but every file the agent opens and every result of a command is new text to read.

#### 17.4.2 Memory for a long context

The context window has to hold the whole request. [05.09](05.09_Making_Models_Smaller.ipynb) gave the memory of the keys and values for every token: 2 × layers × key-value heads × head dimension × 2 bytes. The model's own description has all four numbers:

In [ ]:
info = caspi.ollama("/api/show", {"model": WRITER})["model_info"]
arch = info["general.architecture"]
layers, kv_heads = info[f"{arch}.block_count"], info[f"{arch}.attention.head_count_kv"]
head_dim = info[f"{arch}.embedding_length"] // info[f"{arch}.attention.head_count"]
weights_gb = next(model["size_gb"] for model in caspi.models(info=True) if model["name"] == WRITER)

print(f"{WRITER}: {layers} layers, {kv_heads} key-value heads, {head_dim} dimensions per head")
for tokens in (4096, 16384, 32768, 65536):
    cache_gb = 2 * layers * kv_heads * head_dim * tokens * 2 / 1e9
    print(f"{tokens:6d} tokens: {weights_gb:.1f} GB weights + {cache_gb:4.1f} GB cache = {weights_gb + cache_gb:4.1f} GB")

05.09 put the memory that a model can use on the Pi at about 6 GB. Guides for coding agents recommend a context of at least 32,000 tokens. Compare the two, and the reading speed from Section 17.4.1, and you have the reason why a coding agent on a Pi is an experiment, not a tool.

### 17.5 Your Pi in VS Code

Coding assistants in an editor come in two kinds. An agent works on its own: it reads files, edits them and runs commands, step after step, and every step sends thousands of tokens to the model. Section 17.4 showed what that means on a Pi: minutes of reading before each step, and a context that hardly fits into memory. The other kind keeps the model's job small and leaves the decisions with you: the model suggests, you look, and you accept or not. Each request is a few hundred tokens in and a line or a function out, and that suits the Pi.

We use the second kind. [Continue](https://www.continue.dev), an open-source extension for VS Code, connects your editor to two models on the Pi:

- `qwen2.5-coder:1.5b-base` for **autocomplete**. While you type, it guesses how the line goes on, and the editor shows the guess in grey. It is a base model: it was never trained to chat (05.06), only to continue code. It can also fill in the middle, which means it reads the code before and after the cursor and writes what belongs in between.
- `granite4.2:3b` for the **chat panel**. You ask for a change, it answers with code, and **Apply** writes that code into your file once you have seen the difference and accepted it.

Continue's team moved to Cursor in June 2026, and the extension is no longer developed. Its last version works with Ollama, though, and for this setup we found nothing that works better.

The next cell creates an empty file `wifi.py` next to this notebook and prints the settings for Continue.

In [ ]:
from pathlib import Path

Path("wifi.py").touch()   # creates the file if it is missing, leaves an existing one as it is
print(f"Your file: {Path('wifi.py').resolve()}\n")

print(f"""name: Pi
version: 1.0.0
schema: v1
models:
  - name: Pi autocomplete
    provider: ollama
    model: qwen2.5-coder:1.5b-base
    apiBase: {caspi.OLLAMA}
    roles:
      - autocomplete
    autocompleteOptions:
      modelTimeout: 20000
      debounceDelay: 1000
      maxPromptTokens: 512
  - name: Pi chat
    provider: ollama
    model: granite4.2:3b
    apiBase: {caspi.OLLAMA}
    roles:
      - chat
      - edit
      - apply""")

<div class="alert alert-block alert-warning">

**Free the memory first.** Autocomplete and chat keep two models loaded, about 4.5 GB together. The writer from Section 17.2 does not fit next to either of them, and loading it anyway can freeze the Pi (05.04, Section 4.5.2). Run `caspi.stop()` before you start, and again whenever you switch back to the cells of Sections 17.2 to 17.4.
</div>

<div class="alert alert-block alert-info">

1. In VS Code, open the Extensions view, search for **Continue** and install it.
2. Open Continue in the side bar and open its configuration file, `config.yaml` in the folder `.continue` in your home folder. Replace what is in it with the settings printed above, and save.
3. Run `caspi.stop()` in a cell, as the box above says.
4. Open `wifi.py`, type `def dbm_to_mw(dbm):`, press Enter and wait. After a second or two, a suggestion appears in grey. Press Tab to accept it, or keep typing to ignore it.
5. Open Continue's chat panel and set its mode to **Chat**. Ask: *Add a function channel_of(mhz) to wifi.py that returns the Wi-Fi channel of a frequency in the 2.4 GHz band.* Above the code in the answer, press **Apply**. Continue shows the change in `wifi.py`, and **Accept** keeps it.
</div>

#### 17.5.1 What the settings say

Continue reads everything from `config.yaml`, and you can change it to use other models. Under `models`, the file lists one entry per model:

- `name` is the label Continue shows you. Choose any.
- `provider: ollama` and `apiBase` say where the model runs, here on the Ollama of your Pi. Without `apiBase`, Continue looks for Ollama on your own laptop, at `http://localhost:11434`, which is how you would use a model you installed there.
- `model` is the name of the model, as `caspi.models()` lists it.
- `roles` says what Continue uses the model for: `autocomplete` for the grey suggestions, `chat` for the chat panel, `edit` for changing selected code on request, and `apply` for writing the code of a chat answer into your file. One model can have several roles.
- `autocompleteOptions` matters only for the autocomplete model, and all three settings are there because of the Pi. Continue throws away any suggestion that takes longer than 150 milliseconds, and the Pi needs a second or more, so `modelTimeout` gives it 20 seconds. `debounceDelay` waits until you have stopped typing for a second, so that the Pi does not start on a guess for every key you press. `maxPromptTokens` limits how much of your file Continue sends, because the Pi has to read all of it before it can write the first token.

Not every model can take every role. Autocomplete needs a model that can fill in the middle, which `caspi.show()` lists as `insert`, and on the Pi only the two `qwen2.5-coder` models can. The 7B one writes better code, but it reads about five times slower than the 1.5B one and does not fit into memory next to a chat model. For chat, any chat model on the Pi works: `qwen3.5:4b` knows more than `granite4.2:3b` but answers more slowly.

#### 17.5.2 How fast, and how good?

The next cell asks the autocomplete model directly, the way Continue does: the code before the cursor as the prompt, the code after it as the `suffix`. It tries a short file and a long one.

In [ ]:
AUTOCOMPLETE = "qwen2.5-coder:1.5b-base"


def suggest(before, after="\n"):
    """What autocomplete would show at the cursor, between `before` and `after`."""
    reply = caspi.ask(AUTOCOMPLETE, before, suffix=after, temperature=0, num_predict=64,
                      stop=["\n\n", "\ndef "], show=False)
    print(f"{reply.stats['prompt_tokens']:4d} tokens read, {reply.stats['total_seconds']:4.1f} s:  {reply.text!r}")


caspi.stop()   # the writer from Section 17.2 must not stay next to it
caspi.load(AUTOCOMPLETE)
suggest("# Wi-Fi helpers\n\ndef dbm_to_mw(dbm):\n    ")
suggest(open("caspi.py", encoding="utf-8").read()[:1800] + "\n\ndef dbm_to_mw(dbm):\n    ")

In the short file, the suggestion arrives in a second or two and is right. Put the first 1800 characters of `caspi.py` in front of the same line, and the model reads several hundred tokens before it writes anything, which takes several seconds. In our runs it then also guessed wrong. Autocomplete on the Pi is fast in a small file and slow in a large one, and Continue sends up to 512 tokens of your file with every suggestion.

<div class="alert alert-block alert-info">

Work with it for ten minutes, in `wifi.py` and in a longer file of your own.

- Write a comment that describes a function, such as `# convert a Wi-Fi channel number to its frequency in MHz`, then start the function. Does the suggestion follow your comment?
- Test the `channel_of` that the chat wrote, with the cases of Section 17.2.5: `test(open("wifi.py").read(), "channel_of", CHANNEL_CASES)`. Does it know about channel 14?
- Select a few lines and ask the chat panel what they do.
- Change one model in `config.yaml`, for example `qwen3.5:4b` for chat, save, and ask the chat panel the same question again. Run `caspi.stop()` first.

Note what it got right, what it got wrong and how long you waited.
</div>

This setup suits the Pi. The code never leaves your laptop and the Pi, it works without a network, and it costs nothing per suggestion. Nothing happens on its own: you see every suggestion and every change before it lands in your file. Its limits come from the size of the models and the speed of the Pi. Small models know common patterns and get facts wrong, and neither model tests what it writes; that stays with you, as in Section 17.2. Continue sends only part of your file, not your whole project, and every token it sends costs reading time, which is why the same setup feels quick in a short file and sluggish in a long one. On a larger computer, the same configuration takes larger models: change `model` and `apiBase`, and the rest stays as it is.

### 17.6 Where Vibe Coding Stands

For small programs, prototypes and one-off scripts, vibe coding is fast. People who never learned to program now build their own tools, and programmers hand over the boring parts: boilerplate, first drafts of tests, code in a language they rarely use.

Code that runs is not code that is right, and not code that is safe. Veracode gave more than 100 models 80 coding tasks: in 45 % of the cases the code had a known security flaw, and newer models did no better [04]. In a controlled study by METR, experienced developers working on their own projects were 19 % slower with AI tools, while they believed they had been about 20 % faster [03]. And code that nobody has read is code that nobody can maintain. Tests, reviews and small steps, as in Sections 17.2 and 17.3, are what keep the speed from turning into debt.

The coding agents in the news run on very large models in data centres, with contexts of hundreds of thousands of tokens. The documentation of Cline, a popular open-source coding agent, lists 16 to 32 GB of memory as typical even for small, quantized local models [05]. The best coding model that fits into the Pi's 8 GB, `qwen2.5-coder:7b`, dates from 2024: newer coding models start at around 30 billion parameters. What local coding offers in return is privacy. The code never leaves the device, which matters for a company whose code is its business, and there is no cost per token.

The tools change quickly: Continue, the extension of Section 17.5, stopped being developed sixteen months after Karpathy's post. The ideas in this notebook do not: a loop that runs the code, tests that decide when it is done, and a second look before anything important goes in.

### 17.7 Further Reading and Literature

[01] Karpathy, A. (2025). **The original "vibe coding" post.** X, 2 February 2025. [Post](https://x.com/karpathy/status/1886192184808149383)

[02] Chen, X., Lin, M., Schärli, N., et al. (2023). **Teaching Large Language Models to Self-Debug.** [Paper](https://arxiv.org/abs/2304.05128). Models that fix their code from the results of running it.

[03] Becker, J., Rush, N., Barnes, E., & Rein, D. (2025). **Measuring the Impact of Early-2025 AI on Experienced Open-Source Developer Productivity.** METR. [Paper](https://arxiv.org/abs/2507.09089)

[04] Veracode (2025). **2025 GenAI Code Security Report.** [Press release](https://www.veracode.com/press-release/ai-generated-code-poses-major-security-risks-in-nearly-half-of-all-development-tasks-veracode-research-reveals/)

[05] Cline. **Running models locally.** Documentation. [Website](https://docs.cline.bot/running-models-locally/overview)

[06] Chen, M., Tworek, J., Jun, H., et al. (2021). **Evaluating Large Language Models Trained on Code.** [Paper](https://arxiv.org/abs/2107.03374). Introduced HumanEval: generated code is judged by tests.

[07] Continue. **Autocomplete.** Documentation of the extension used in Section 17.5. [Website](https://docs.continue.dev/customize/deep-dives/autocomplete)

---
### 17.8 Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

Use your own outputs as evidence. Copy the relevant numbers into your answer so that they remain visible after saving.

**Question 01: Prompting and vibe coding**

In your own words: how does vibe coding differ from asking a model a question? Use one step of Section 17.2 as an example.

(answer here)

**Question 02: Running generated code**

Why is `exec` the step that makes vibe coding work, and why is it the dangerous one? Name two ways to reduce the risk.

(answer here)

**Question 03: The missing fact**

What did the model do with the channel 14 error in Section 17.2.5? Paste the line that computes the channel in each version, and the sentence you added. What did the model need that the error message did not give it?

(answer here)

**Question 04: Your loop**

Paste your `vibe` function and the rounds it needed for `parse_log`. Where did the early versions go wrong, and what did you add to the task to get it right?

(answer here)

**Question 05: Few words**

Paste your shortest task for `payoff`. What made so few words enough, and what is the risk of relying on that?

(answer here)

**Question 06: Tests and reviews**

What did the reviewer say about `average_signal`, and was it right? Did your `vibe_reviewed` make the log parser better? What can a review find that a test cannot, and the other way round?

(answer here)

**Question 07: The Pi's limits**

Report the reading speed and the memory table of Section 17.4. Which context window would you give a coding agent on the Pi, and how long would its first answer take?

(answer here)

**Question 08: Autocomplete on the Pi**

Report the two measurements of Section 17.5: tokens read, seconds and suggestion. What did autocomplete and the chat with Apply get right and wrong in your own tries, and how long did you wait? Did `channel_of` pass the tests? For which kind of work would you use this setup on a Pi, and for which not?

(answer here)

**Question 09: Your verdict**

Where would you use vibe coding with a local model, and where not? Use at least one fact from Section 17.6.

(answer here)

**Question 10: Optional: the same task in the cloud**

Give the task from Section 17.5 to a cloud coding assistant. Compare the time, the result and what you had to send to the cloud.

(answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)